# 03 — CNN Prototyping: Transfer Learning for DR Classification

This notebook prototypes the CNN model using transfer learning.

**Goals:**
- Build and inspect the transfer learning architecture
- Train on a small subset for quick validation
- Compare backbones (ResNet50, EfficientNetB0)
- Visualize training curves

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import load_settings
from src.models.retinal_cnn import (
    build_cnn_model,
    unfreeze_and_fine_tune,
    get_callbacks,
)

settings = load_settings()
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Build the Model Architecture

In [ ]:
# Build with default backbone from settings (EfficientNetB0)
model = build_cnn_model()
model.summary()

In [ ]:
# Inspect layer counts
print(f'Total layers: {len(model.layers)}')
print(f'Trainable parameters: {model.count_params():,}')

trainable_count = sum(np.prod(v.shape) for v in model.trainable_weights)
non_trainable_count = sum(np.prod(v.shape) for v in model.non_trainable_weights)
print(f'Trainable weights: {trainable_count:,}')
print(f'Non-trainable weights: {non_trainable_count:,}')

## 2. Create Synthetic Data for Prototyping

In [ ]:
# Generate synthetic data to verify the pipeline works end-to-end
target_size = settings['image']['target_size']
num_classes = settings['cnn']['num_classes']

n_train, n_val = 64, 16
X_train = np.random.rand(n_train, target_size[1], target_size[0], 3).astype(np.float32)
y_train = np.random.randint(0, num_classes, n_train)
X_val = np.random.rand(n_val, target_size[1], target_size[0], 3).astype(np.float32)
y_val = np.random.randint(0, num_classes, n_val)

print(f'Synthetic train: {X_train.shape}, Val: {X_val.shape}')

## 3. Phase 1 — Train with Frozen Backbone

In [ ]:
model = build_cnn_model(freeze_base=True)

history1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=16,
    verbose=1,
)

print('Phase 1 complete.')

## 4. Phase 2 — Fine-Tune Backbone

In [ ]:
model = unfreeze_and_fine_tune(model, learning_rate=1e-5)

history2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    initial_epoch=5,
    batch_size=16,
    verbose=1,
)

print('Phase 2 (fine-tuning) complete.')

## 5. Training Curves

In [ ]:
# Combine histories
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
epochs_range = range(1, len(loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_range, loss, 'b-', label='Train Loss')
axes[0].plot(epochs_range, val_loss, 'r-', label='Val Loss')
axes[0].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(epochs_range, acc, 'b-', label='Train Accuracy')
axes[1].plot(epochs_range, val_acc, 'r-', label='Val Accuracy')
axes[1].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('Training Curves (Frozen → Fine-Tuned)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Compare Backbones

In [ ]:
backbones = ['EfficientNetB0', 'ResNet50']
results = {}

for backbone in backbones:
    print(f'\n--- Testing {backbone} ---')
    m = build_cnn_model(backbone_name=backbone, freeze_base=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=3, batch_size=16, verbose=0)
    val_acc_final = h.history['val_accuracy'][-1]
    params = m.count_params()
    results[backbone] = {'val_accuracy': val_acc_final, 'params': params}
    print(f'{backbone}: val_acc={val_acc_final:.4f}, params={params:,}')

print('\n--- Backbone Comparison ---')
for name, res in results.items():
    print(f'{name:20s}  val_acc={res["val_accuracy"]:.4f}  params={res["params"]:>12,}')

## Summary

- Transfer learning pipeline works end-to-end
- Two-phase strategy (frozen → fine-tuned) validates correctly
- EfficientNetB0 offers a good balance of accuracy and parameters
- Ready to scale to full dataset using `src/pipeline/train.py`